In [1]:
import pandas as pd
from sqlalchemy import text


In [2]:
def load_user_history(engine, user_id):
    """
    DB에서 특정 user_id의 러닝 기록을 불러와 XAI와 LSTM이 공통으로 사용 가능하도록 만드는 함수
    """
    query = text("""
        SELECT
            user_id,
            start_time,
            distance_km,
            pace_km,
            avg_heart_rate,
            TIME_TO_SEC(duration_time) AS duration_sec
        FROM running_record
        WHERE user_id = :uid
        ORDER BY start_time
    """)
    
    df = pd.read_sql(query, engine, params={"uid": user_id})
    df["start_time"] = pd.to_datetime(df["start_time"])
    return df


In [3]:
def split_recent_and_previous(df, recent_n=5):
    """
    최근 recent_n개와 그 이전 구간을 나누는 함수
    """
    if len(df) <= recent_n:
        return df, pd.DataFrame()  # 이전 데이터 없음
    
    recent = df.iloc[-recent_n:]
    previous = df.iloc[:-recent_n]
    return recent, previous


In [4]:
def generate_running_explanation(recent, previous, next_km=None):
    """
    최근 패턴 vs 이전 패턴 비교 + LSTM 예측값(next_km) 기반 설명 생성
    """
    explanations = []
    
    # ----------------------------
    # 거리 변화 분석
    # ----------------------------
    if not previous.empty:
        recent_dist = recent["distance_km"].mean()
        prev_dist = previous["distance_km"].mean()

        if recent_dist > prev_dist:
            explanations.append(f"최근 평균 거리({recent_dist:.1f}km)가 이전({prev_dist:.1f}km)보다 증가했습니다. 체력이 향상 중이에요! 💪")
        else:
            explanations.append(f"최근 평균 거리({recent_dist:.1f}km)가 이전({prev_dist:.1f}km)보다 감소했습니다. 회복 중이거나 일정이 바빴을 가능성이 있어요.")

    # ----------------------------
    # 페이스 변화 분석
    # ----------------------------
    if not previous.empty:
        recent_pace = recent["pace_km"].mean()
        prev_pace = previous["pace_km"].mean()

        if recent_pace < prev_pace:
            explanations.append(f"최근 페이스({recent_pace:.2f} min/km)가 더 빨라졌어요. 달리기 효율이 좋아지고 있습니다! 🏃‍♂️💨")
        else:
            explanations.append(f"최근 페이스({recent_pace:.2f} min/km)가 느려졌어요. 무리 없이 안정적인 러닝을 하고 있습니다.")

    # ----------------------------
    # 심박 변화 분석
    # ----------------------------
    if not previous.empty:
        recent_hr = recent["avg_heart_rate"].mean()
        prev_hr = previous["avg_heart_rate"].mean()

        if recent_hr < prev_hr:
            explanations.append(f"평균 심박수({recent_hr:.0f} bpm)가 감소했어요. 동일한 페이스에서도 여유가 생긴 상태입니다. 🧘‍♂️")
        else:
            explanations.append(f"평균 심박수({recent_hr:.0f} bpm)가 조금 상승했어요. 훈련 강도가 올라간 것으로 보입니다.")

    # ----------------------------
    # LSTM 예측 기반 추가 설명
    # ----------------------------
    if next_km is not None:
        explanations.append(f"다음 운동 예상 거리는 약 {next_km:.1f} km 입니다. 이 흐름을 잘 유지해 보세요! 🔥")

    return "\n".join(explanations)


In [ ]:
def explain_running_pattern_from_db(user_id, engine, next_km=None):
    """
    DB에서 유저 기록을 불러와 XAI 설명을 생성하는 최종 함수
    """
    # 1) DB 로드
    df = load_user_history(engine, user_id)
    if df.empty:
        return "해당 유저의 러닝 기록이 없습니다."

    # 2) 최근/과거 데이터 분리
    recent, previous = split_recent_and_previous(df, recent_n=5)

    # 3) 설명 생성
    explanation = generate_running_explanation(
        recent=recent,
        previous=previous,
        next_km=next_km
    )

    return explanation


In [6]:
def predict_next_distance_for_user(user_id, df, scaler, model, seq_len=7):
    # 특정 유저의 기록만 가져오기
    user_df = df[df['user_id'] == user_id].sort_values('start_time')

    # 데이터가 충분하지 않은 경우 처리
    if len(user_df) < seq_len:
        raise ValueError(f"User {user_id} has only {len(user_df)} records, but seq_len={seq_len} required.")

    # 최근 seq_len개만 가져오기
    recent = user_df.iloc[-seq_len:]
    X_recent = recent[['distance_km', 'pace_km', 'avg_heart_rate', 'duration_sec']].values  

    # 스케일링 적용
    X_recent_scaled = scaler.transform(X_recent)

    # LSTM 입력 형태로 변환
    X_input = X_recent_scaled.reshape(1, seq_len, X_recent.shape[1])

    # 예측
    y_pred = model.predict(X_input)[0, 0]

    return y_pred


In [7]:
next_km = predict_next_distance_for_user(
    user_id=1,
    df=df,            # → 여기서 df 대신 DB에서 읽기 가능
    scaler=scaler,
    model=model
)


NameError: name 'df' is not defined

In [ ]:
result = explain_running_pattern_from_db(
    user_id=1,
    engine=engine,
    next_km=next_km   # ← LSTM 결과 넣기
)

print(result)
